In [2]:
import glob, os
import pandas as pd
import numpy as np

In [3]:
data = pd.read_csv(r"C:\Users\Colin\git\hackathon-237858\raw\raw\CMSHospital\Hospital_General_Information.csv")

In [4]:
data.head()

,Facility ID,Facility Name,Address,City/Town,State,ZIP Code,County/Parish,Telephone Number,Hospital Type,Hospital Ownership,...,Count of READM Measures Better,Count of READM Measures No Different,Count of READM Measures Worse,READM Group Footnote,Pt Exp Group Measure Count,Count of Facility Pt Exp Measures,Pt Exp Group Footnote,TE Group Measure Count,Count of Facility TE Measures,TE Group Footnote
0,010001,SOUTHEAST HEALTH MEDICAL CENTER,1108 ROSS CLARK CIRCLE,DOTHAN,AL,36301,HOUSTON,(334) 793-8701,Acute Care Hospitals,Government - Hospital District or Authority,...,1,9,1,NaN,15,15,NaN,10,10,NaN
1,010005,MARSHALL MEDICAL CENTERS,2505 U S HIGHWAY 431 NORTH,BOAZ,AL,35957,MARSHALL,(256) 593-8310,Acute Care Hospitals,Government - Hospital District or Authority,...,1,8,0,NaN,15,15,NaN,10,10,NaN
2,010006,NORTH ALABAMA MEDICAL CENTER,1701 VETERANS DRIVE,FLORENCE,AL,35630,LAUDERDALE,(256) 768-8400,Acute Care Hospitals,Proprietary,...,1,8,0,NaN,15,15,NaN,10,9,NaN
3,010007,MIZELL MEMORIAL HOSPITAL,702 N MAIN ST,OPP,AL,36467,COVINGTON,(334) 493-3541,Acute Care Hospitals,Voluntary non-profit - Private,...,0,3,2,NaN,15,5,NaN,10,7,NaN
4,010011,ST. VINCENT'S EAST,50 MEDICAL PARK EAST DRIVE,BIRMINGHAM,AL,35235,JEFFERSON,(205) 838-3122,Acute Care Hospitals,Voluntary non-profit - Private,...,1,5,2,29.0,15,10,29.0,10,7,29.0


In [5]:
batch_df = data[["Facility ID", "Address", "City/Town", "State", "ZIP Code"]].copy()
batch_df.columns = ["id", "street", "city", "state", "zip"]
batch_df.to_csv("addresses.csv", index=False, header=False)

In [6]:
output = pd.read_csv(
    "geocoderresult.csv",
    header=None,
    names=[
        "id",
        "input_address",
        "match_status",
        "match_type",
        "matched_address",
        "coordinates",
        "tiger_line_id",
        "tiger_line_side",
        "state_fips",
        "county_fips",
        "tract",
        "block",
    ],
)

output.head()


,id,input_address,match_status,match_type,matched_address,coordinates,tiger_line_id,tiger_line_side,state_fips,county_fips,tract,block
0,170208,"7820 W 165TH STREET, OVERLAND PARK, KS, 66223",Match,Exact,"7820 W 165TH ST, OVERLAND PARK, KS, 66223","-94.67920726456,38.831651182154",654376720.0,L,20.0,91.0,53418.0,2001.0
1,170203,"1619 K 66, GALENA, KS, 66739",No_Match,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,43005F,"2501 WEST 22ND STREET, SIOUX FALLS, SD, 57117",Match,Non_Exact,"2501 W 22ND ST, SIOUX FALLS, SD, 57105","-96.757638949529,43.532987962874",34971391.0,L,46.0,99.0,1200.0,2000.0
3,170200,"14800 WEST TERESA ST, WICHITA, KS, 67235",Match,Non_Exact,"14800 W SAINT TERESA ST, WICHITA, KS, 67235","-97.512441828541,37.724727188724",645801654.0,L,20.0,173.0,9507.0,2000.0
4,340008,"500 LAUCHWOOD DR, LAURINBURG, NC, 28352",Match,Exact,"500 LAUCHWOOD DR, LAURINBURG, NC, 28352","-79.470539951351,34.75177249798",25793564.0,L,37.0,165.0,10300.0,1055.0


In [7]:
failure_rate = (output["match_status"] == "No_Match").mean()

print(f"Failure rate: {failure_rate:.2%}")

Failure rate: 13.69%


In [11]:
# Read the Census geocoder output
df = pd.read_csv(
    "geocoderresult.csv",
    header=None,
    names=[
        "id", "input_address", "match_status", "match_type",
        "matched_address", "coordinates", "tiger_line_id",
        "tiger_line_side", "state_fips", "county_fips",
        "tract", "block"
    ]
)
df.head()

,id,input_address,match_status,match_type,matched_address,coordinates,tiger_line_id,tiger_line_side,state_fips,county_fips,tract,block
0,170208,"7820 W 165TH STREET, OVERLAND PARK, KS, 66223",Match,Exact,"7820 W 165TH ST, OVERLAND PARK, KS, 66223","-94.67920726456,38.831651182154",654376720.0,L,20.0,91.0,53418.0,2001.0
1,170203,"1619 K 66, GALENA, KS, 66739",No_Match,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,43005F,"2501 WEST 22ND STREET, SIOUX FALLS, SD, 57117",Match,Non_Exact,"2501 W 22ND ST, SIOUX FALLS, SD, 57105","-96.757638949529,43.532987962874",34971391.0,L,46.0,99.0,1200.0,2000.0
3,170200,"14800 WEST TERESA ST, WICHITA, KS, 67235",Match,Non_Exact,"14800 W SAINT TERESA ST, WICHITA, KS, 67235","-97.512441828541,37.724727188724",645801654.0,L,20.0,173.0,9507.0,2000.0
4,340008,"500 LAUCHWOOD DR, LAURINBURG, NC, 28352",Match,Exact,"500 LAUCHWOOD DR, LAURINBURG, NC, 28352","-79.470539951351,34.75177249798",25793564.0,L,37.0,165.0,10300.0,1055.0


In [14]:
# Keep only matched addresses
matched_df = df[df["match_status"] == "Match"].copy()

# Save for the pipeline
matched_df.to_csv("matched_addresses.csv", index=False,)

print(f"Matched: {len(matched_df)} / {len(df)}")
matched_df.head()

Matched: 4631 / 5419


,id,input_address,match_status,match_type,matched_address,coordinates,tiger_line_id,tiger_line_side,state_fips,county_fips,tract,block
0,170208,"7820 W 165TH STREET, OVERLAND PARK, KS, 66223",Match,Exact,"7820 W 165TH ST, OVERLAND PARK, KS, 66223","-94.67920726456,38.831651182154",654376720.0,L,20.0,91.0,53418.0,2001.0
2,43005F,"2501 WEST 22ND STREET, SIOUX FALLS, SD, 57117",Match,Non_Exact,"2501 W 22ND ST, SIOUX FALLS, SD, 57105","-96.757638949529,43.532987962874",34971391.0,L,46.0,99.0,1200.0,2000.0
3,170200,"14800 WEST TERESA ST, WICHITA, KS, 67235",Match,Non_Exact,"14800 W SAINT TERESA ST, WICHITA, KS, 67235","-97.512441828541,37.724727188724",645801654.0,L,20.0,173.0,9507.0,2000.0
4,340008,"500 LAUCHWOOD DR, LAURINBURG, NC, 28352",Match,Exact,"500 LAUCHWOOD DR, LAURINBURG, NC, 28352","-79.470539951351,34.75177249798",25793564.0,L,37.0,165.0,10300.0,1055.0
5,170204,"3251 NORTH ROCK ROAD, DERBY, KS, 67037",Match,Non_Exact,"3251 S ROCK RD, DERBY, KS, 67037","-97.244464928881,37.582631433602",656603083.0,R,20.0,173.0,10007.0,1021.0


# Hiccup: Duplicates found in the two datasets

# Part 2: Accurate locations for hospitals and other key facilities

The first pass above geocoded only the **Hospital General Information** (HGI) addresses, and 13.69% (788 / 5,419) came back `No_Match`. Also, a `Match` is not always correct: a `Non_Exact` match can move a hospital to a different street (e.g. `3251 NORTH ROCK ROAD` became `3251 S ROCK RD`).

This part:
1. Reads every CMS facility file in `raw/raw/CMSFacilities`. **Provider of Services (POS)** is the backbone: it lists every Medicare-certified facility with its CCN, category, bed count, and CMS's own state/county FIPS codes.
2. Separates **hospitals** (acute, critical access, psychiatric, rehab, long-term, children's, rural emergency, VA/DoD) from **other facilities**. It keeps FQHCs and Rural Health Clinics, which are the safety-net primary-care sites tied to the HRSA HPSA/MUA data. It drops SNFs, home health, hospice, dialysis, transplant programs and similar facilities, which don't provide hospital access.
3. Collects **every address CMS has for each facility** (HGI, POS, Medicare enrollment) and geocodes all of them with the Census batch geocoder. A facility that fails on one address often matches on another.
4. Keeps a coordinate only if it passes accuracy checks. It needs a Census `Match` whose state and county agree with CMS's own FIPS codes. A `Non_Exact` match must also keep the same house number and street direction.
5. Facilities with no accurate coordinate are the **outliers**. They're written to their own CSV with the reason, and are **not** plotted.

Outputs (in `addresses/output/`): `hospital_locations.csv`, `other_facility_locations.csv`, `facility_location_outliers.csv`. These are historical address-research outputs; the production map is served by `src/optimal_hospital_placer/api/` from the production ETL artifacts.

In [ ]:
import csv, hashlib, io, json, math, re, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import requests

# Find the repo root from wherever the notebook is run.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "raw" / "raw").is_dir())
RAW = ROOT / "raw" / "raw"
FAC = RAW / "CMSFacilities"
OUT = ROOT / "addresses" / "output"
CACHE = ROOT / "addresses" / "geocode_cache"
OUT.mkdir(exist_ok=True)
CACHE.mkdir(exist_ok=True)

def read(path, **kw):
    return pd.read_csv(path, dtype=str, encoding="latin1", keep_default_na=False, **kw)

pos_all = read(FAC / "Hospital_and_other.DATA.Q2_2026.csv")
hgi = read(RAW / "CMSHospital" / "Hospital_General_Information.csv")
enroll = {
    "hospital": read(FAC / "Hospital_Enrollments_2026.07.31.csv"),
    "fqhc": read(FAC / "FQHC_Enrollments_2026.07.17.csv"),
    "rhc": read(FAC / "RHC_Enrollments_2026.07.17.csv"),
}
print(f"POS rows: {len(pos_all):,}   HGI hospitals: {len(hgi):,}   " +
      "   ".join(f"{k} enrollments: {len(v):,}" for k, v in enroll.items()))

### Separate hospitals from other facilities

POS `PGM_TRMNTN_CD == "00"` means the facility is still active. Category `01` is hospitals, `21` FQHCs and `12` Rural Health Clinics. Hospital subtype codes are mapped to the same type names HGI uses, so the map's type filter stays consistent. Two subtypes are dropped: `20` is transplant programs, which have a separate CCN (`xx98xx`) but sit inside their host hospital, and `03` is religious non-medical institutions.

In [ ]:
HOSPITAL_SUBTYPES = {
    "01": "Acute Care Hospitals",
    "11": "Critical Access Hospitals",
    "28": "Rural Emergency Hospital",
    "04": "Psychiatric", "24": "Psychiatric", "25": "Psychiatric",
    "06": "Childrens",
    "02": "Long-term",
    "05": "Rehabilitation",
}
OTHER_CATEGORIES = {"21": "Federally Qualified Health Center", "12": "Rural Health Clinic"}

# CMS's own state -> state FIPS (used to check the geocoder didn't land in another state).
STATE_FIPS = (pos_all[pos_all.FIPS_STATE_CD != ""].groupby("STATE_CD").FIPS_STATE_CD
              .agg(lambda s: s.mode().iat[0]).to_dict())

pos = pos_all[(pos_all.PGM_TRMNTN_CD == "00") & pos_all.STATE_CD.isin(STATE_FIPS)].copy()
pos["fac_type"] = np.where(
    pos.PRVDR_CTGRY_CD == "01", pos.PRVDR_CTGRY_SBTYP_CD.map(HOSPITAL_SUBTYPES), pos.PRVDR_CTGRY_CD.map(OTHER_CATEGORIES))
pos = pos[pos.fac_type.notna()]
# POS rows by CCN in any status, for county FIPS / beds of HGI hospitals that aren't active in POS
pos_by_ccn = pos_all.drop_duplicates("PRVDR_NUM").set_index("PRVDR_NUM")

# Hospitals = active POS hospitals + everything in HGI (VA and DoD hospitals are only in HGI).
hgi_idx = hgi.set_index("Facility ID")
hosp_ccns = sorted(set(pos.loc[pos.PRVDR_CTGRY_CD == "01", "PRVDR_NUM"]) | set(hgi_idx.index))
other = pos[pos.PRVDR_CTGRY_CD != "01"]

def facility_row(ccn, fac_class):
    p = pos_by_ccn.loc[ccn] if ccn in pos_by_ccn.index else None
    h = hgi_idx.loc[ccn] if ccn in hgi_idx.index else None
    in_pos_active = p is not None and ccn in set_pos_active
    return {
        "ccn": ccn,
        "facility_class": fac_class,
        "name": (h["Facility Name"] if h is not None else p["FAC_NAME"]).strip(),
        "type": h["Hospital Type"] if h is not None else pos_type[ccn],
        "state": h["State"] if h is not None else p["STATE_CD"],
        "state_fips": STATE_FIPS.get(h["State"] if h is not None else p["STATE_CD"], ""),
        "county_fips": (p["FIPS_STATE_CD"] + p["FIPS_CNTY_CD"]) if p is not None and p["FIPS_CNTY_CD"] else "",
        "county": h["County/Parish"].strip() if h is not None else "",
        "address": (h["Address"] if h is not None else p["ST_ADR"]).strip(),
        "city": (h["City/Town"] if h is not None else p["CITY_NAME"]).strip(),
        "zip": (h["ZIP Code"] if h is not None else p["ZIP_CD"]).strip().zfill(5)[:5],
        "phone": h["Telephone Number"] if h is not None else p["PHNE_NUM"],
        "ownership": h["Hospital Ownership"] if h is not None else "",
        "emergency": (h["Emergency Services"] == "Yes") if h is not None else None,
        "rating": h["Hospital overall rating"] if h is not None and h["Hospital overall rating"].isdigit() else "",
        "beds": p["BED_CNT"] if p is not None else "",
        "in_hgi": h is not None,
        "in_pos_active": in_pos_active,
    }

set_pos_active = set(pos.PRVDR_NUM)
pos_type = pos.set_index("PRVDR_NUM").fac_type.to_dict()
facilities = pd.DataFrame(
    [facility_row(c, "hospital") for c in hosp_ccns] +
    [facility_row(c, "other") for c in other.PRVDR_NUM]
)
# Only HGI hospitals with a known state (drops territories outside the POS state list, if any)
facilities = facilities[facilities.state_fips != ""].reset_index(drop=True)
print(facilities.groupby(["facility_class", "type"]).size().to_string())
print(f"\nHospitals: {(facilities.facility_class == 'hospital').sum():,} "
      f"({facilities.in_hgi.sum():,} in HGI)   Other facilities: {(facilities.facility_class == 'other').sum():,}")

### Every address CMS has for each facility

Sources, in order of trust: HGI, then POS, then the Medicare enrollment files (for hospitals, the `MAIN/PRIMARY` location). PO boxes and suite/floor text are stripped out because the geocoder can't place them. Identical addresses are geocoded only once.

In [ ]:
PO_BOX = re.compile(r"\b(P\.?\s?O\.?\s*BOX|BOX)\s*\d+\b[\s,/-]*", re.I)
UNIT = re.compile(r"\b(SUITE|STE|FLOOR|FL|BLDG|BUILDING|ROOM|RM|UNIT)\b\.?\s*#?[\w-]*|#\s*\w+", re.I)

def clean_street(street):
    s = PO_BOX.sub(" ", street or "")
    s = UNIT.sub(" ", s)
    s = re.sub(r"^\s*ONE\b", "1", s, flags=re.I)
    s = re.sub(r"^\s*TWO\b", "2", s, flags=re.I)
    return re.sub(r"\s+", " ", s).strip(" ,-/").upper()

cands = []  # (ccn, source, street, city, state, zip)
fac_idx = facilities.set_index("ccn")
for ccn, h in hgi_idx.iterrows():
    cands.append((ccn, "hgi", h["Address"], h["City/Town"], h["State"], h["ZIP Code"]))
for _, p in pos[pos.PRVDR_NUM.isin(fac_idx.index)].iterrows():
    cands.append((p.PRVDR_NUM, "pos", p.ST_ADR, p.CITY_NAME, p.STATE_CD, p.ZIP_CD))
for kind, e in enroll.items():
    if kind == "hospital":
        e = e[e["PRACTICE LOCATION TYPE"].str.startswith("MAIN")]
    for _, r in e[e.CCN.isin(fac_idx.index)].iterrows():
        cands.append((r.CCN, "enrollment", r["ADDRESS LINE 1"], r.CITY, r.STATE, r["ZIP CODE"]))
        if r["ADDRESS LINE 2"]:  # sometimes line 1 is a PO box / building name and line 2 the street
            cands.append((r.CCN, "enrollment", r["ADDRESS LINE 2"], r.CITY, r.STATE, r["ZIP CODE"]))

cands = pd.DataFrame(cands, columns=["ccn", "source", "street", "city", "state", "zip"])
cands["street"] = cands.street.map(clean_street)
cands["city"] = cands.city.str.strip().str.upper()
cands["zip"] = cands.zip.str.strip().str.zfill(5).str[:5]
cands = cands[cands.street.str.contains(r"\d") | cands.street.str.len().gt(5)]  # drop blanks / bare PO boxes
cands = cands.drop_duplicates(["ccn", "street", "city", "state", "zip"]).reset_index(drop=True)

addr_key = cands.street + "|" + cands.city + "|" + cands.state + "|" + cands.zip
cands["addr_id"] = addr_key.map(lambda k: hashlib.md5(k.encode()).hexdigest()[:12])
unique_addr = cands.drop_duplicates("addr_id")[["addr_id", "street", "city", "state", "zip"]]
print(f"{len(cands):,} candidate addresses for {cands.ccn.nunique():,} facilities; {len(unique_addr):,} unique to geocode")
print(cands.groupby("ccn").size().value_counts().sort_index().rename("facilities by # of addresses").to_string())

### Census batch geocoding (cached)

Same service and benchmark as the first pass (`geographies/addressbatch`, current benchmark and vintage), sent in chunks of 1,000 across 4 threads. Each answer is cached in `addresses/geocode_cache/` by address id, so a rerun only sends new addresses and works offline once the cache is complete.

In [ ]:
GEOCODER = "https://geocoding.geo.census.gov/geocoder/geographies/addressbatch"
COLS = ["addr_id", "input_address", "match_status", "match_type", "matched_address",
        "coordinates", "tiger_line_id", "tiger_line_side", "state_fips_g", "county_fips_g", "tract", "block"]
CACHE_FILE = CACHE / "census_results.csv"

cached = pd.read_csv(CACHE_FILE, dtype=str, keep_default_na=False) if CACHE_FILE.exists() else pd.DataFrame(columns=COLS)
todo = unique_addr[~unique_addr.addr_id.isin(cached.addr_id)]
print(f"cached: {len(cached):,}   to geocode: {len(todo):,}")

def geocode_chunk(chunk, tries=4):
    buf = io.StringIO()
    chunk.to_csv(buf, index=False, header=False)
    for attempt in range(tries):
        try:
            r = requests.post(GEOCODER, data={"benchmark": "Public_AR_Current", "vintage": "Current_Current"},
                              files={"addressFile": ("addresses.csv", buf.getvalue(), "text/csv")}, timeout=900)
            r.raise_for_status()
            rows = [row + [""] * (len(COLS) - len(row)) for row in csv.reader(io.StringIO(r.text)) if row]
            return pd.DataFrame(rows, columns=COLS)
        except (requests.RequestException, ValueError) as exc:
            print(f"  chunk failed ({exc}); retry {attempt + 1}")
            time.sleep(10 * (attempt + 1))
    return pd.DataFrame(columns=COLS)

chunks = [todo.iloc[i:i + 1000] for i in range(0, len(todo), 1000)]
t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:
    for i, res in enumerate(pool.map(geocode_chunk, chunks), 1):
        cached = pd.concat([cached, res], ignore_index=True).drop_duplicates("addr_id", keep="last")
        cached.to_csv(CACHE_FILE, index=False)  # save after every chunk so an interrupted run resumes
        print(f"  chunk {i}/{len(chunks)} done ({time.time() - t0:.0f}s)")

geo = cached[cached.addr_id.isin(unique_addr.addr_id)]
print(geo.match_status.value_counts().to_string())

### Accuracy checks

A geocoded address counts as **accurate** only if all of these hold:
* Census returned `Match`, not `No_Match` or `Tie`.
* The matched **state** equals the facility's state.
* The matched **county** equals CMS's county FIPS from POS, when POS has one. Connecticut is skipped because Census now uses planning regions there instead of the counties POS lists. The one exception is an `Exact` match that also keeps the same ZIP. Checking the misses showed POS's county is usually the wrong one there: towns on a county line, or old county codes such as Alaska's split census areas.
* An `Exact` match passes. A `Non_Exact` match must keep the **same house number** and must not flip a street direction (N↔S, E↔W). Retrying failed addresses without the ZIP produced exactly these errors (`934 SOUTH GLOSTER` became `934 N GLOSTER`, `2300 WEST BROAD` became `2300 E BROAD`).

In [ ]:
DIRS = {"NORTH": "N", "SOUTH": "S", "EAST": "E", "WEST": "W", "NORTHEAST": "NE", "NORTHWEST": "NW",
        "SOUTHEAST": "SE", "SOUTHWEST": "SW"}

def house_number(street):
    m = re.match(r"\s*(\d+)", street or "")
    return m.group(1) if m else None

def directions(street):
    words = re.sub(r"[^A-Z ]", " ", (street or "").upper()).split()
    return {DIRS.get(w, w) for w in words if DIRS.get(w, w) in {"N", "S", "E", "W", "NE", "NW", "SE", "SW"}}

g = cands.merge(geo, on="addr_id", how="left").merge(
    facilities[["ccn", "facility_class", "state_fips", "county_fips"]], on="ccn")
g[["lon", "lat"]] = g.coordinates.str.split(",", expand=True).reindex(columns=[0, 1]).apply(pd.to_numeric, errors="coerce").values
g["matched_street"] = g.matched_address.str.split(",").str[0]
g["state_fips_g"] = g.state_fips_g.str.zfill(2)
g["county_geo"] = g.state_fips_g + g.county_fips_g.str.zfill(3)

def verdict(r):
    if r.match_status != "Match" or pd.isna(r.lat):
        return "no_match" if r.match_status in ("No_Match", "", None) or pd.isna(r.match_status) else r.match_status.lower()
    if r.state_fips_g != r.state_fips:
        return "wrong_state"
    if r.county_fips and r.state_fips != "09" and r.county_geo != r.county_fips:
        if not (r.match_type == "Exact" and r.matched_address.rsplit(", ", 1)[-1] == r.zip):
            return "wrong_county"
    if r.match_type == "Exact":
        return "ok"
    hn_in, hn_out = house_number(r.street), house_number(r.matched_street)
    if not hn_in or hn_in != hn_out:
        return "non_exact_house_number"
    d_in, d_out = directions(r.street), directions(r.matched_street)
    if d_in and d_out and d_in != d_out:
        return "non_exact_direction"
    return "ok"

g["verdict"] = g.apply(verdict, axis=1)
print(g.verdict.value_counts().to_string())

### Fallback 1: resolve Census "Tie" results by county

A `Tie` means the address exists more than once, e.g. N and S New Ballas Rd. The one-line geocoder lists every candidate with its county. If exactly one candidate lies in CMS's county, it's the facility.

In [ ]:
ONELINE = "https://geocoding.geo.census.gov/geocoder/geographies/onelineaddress"
COORDS = "https://geocoding.geo.census.gov/geocoder/geographies/coordinates"
GEO_PARAMS = {"benchmark": "Public_AR_Current", "vintage": "Current_Current", "layers": "Counties", "format": "json"}

def json_cache(name):
    path = CACHE / name
    return path, (json.loads(path.read_text()) if path.exists() else {})

def get_json(url, params, tries=4):
    for attempt in range(tries):
        try:
            r = requests.get(url, params=params, timeout=60)
            r.raise_for_status()
            return r.json()
        except (requests.RequestException, ValueError):
            time.sleep(3 * (attempt + 1))
    return None

def tie_candidates(address):
    d = get_json(ONELINE, {**GEO_PARAMS, "address": address})
    if d is None:
        return None
    return [{"lat": m["coordinates"]["y"], "lon": m["coordinates"]["x"], "matched": m["matchedAddress"],
             "county": m["geographies"]["Counties"][0]["GEOID"]} for m in d["result"]["addressMatches"]]

ties = g[g.verdict == "tie"]
tie_path, tie_cache = json_cache("census_ties.json")
todo = sorted(set(ties.input_address) - set(tie_cache))
with ThreadPoolExecutor(max_workers=8) as pool:
    for address, result in zip(todo, pool.map(tie_candidates, todo)):
        if result is not None:
            tie_cache[address] = result
tie_path.write_text(json.dumps(tie_cache))

resolved = 0
for i, r in ties.iterrows():
    want = r.county_fips if r.county_fips and r.state_fips != "09" else None
    hits = [c for c in tie_cache.get(r.input_address, [])
            if (c["county"] == want if want else c["county"][:2] == r.state_fips)]
    if len(hits) == 1:
        c = hits[0]
        g.loc[i, ["lat", "lon", "matched_address", "match_type", "verdict"]] = [c["lat"], c["lon"], c["matched"], "Tie_resolved", "ok"]
        resolved += 1
print(f"Ties resolved by county: {resolved} / {len(ties)}")

### Fallback 2: find hospitals the Census street map doesn't have in OpenStreetMap

Most remaining hospital failures are addresses missing from the Census street map: rural highway numbers, new streets, campus names. OpenStreetMap maps most hospitals as buildings. For each hospital still unplaced, look within 12 miles of its ZIP's centre for an OSM hospital with a matching name. This reuses `web/legacy_prototype/geocode_hospitals.py`, an archived prototype utility: word-overlap ≥ 0.5 after dropping generic words like HOSPITAL and MEDICAL, and one OSM hospital can't be claimed by two differently named CMS hospitals. A match is kept only if Census places the OSM point in **CMS's county**. The one exception is a word-for-word name match in the same state. Mercy Hospital Joplin, for example, sits on the Jasper/Newton county line, while weak matches such as a lone "VA" still need the county to agree. This is hospitals only; OSM's clinic coverage and generic clinic names are too unreliable for FQHCs and RHCs.

In [ ]:
import sys
sys.path.insert(0, str(ROOT / "web" / "legacy_prototype"))
from geocode_hospitals import best_osm_match, name_tokens, osm_hospitals_near

placed = set(g.loc[g.verdict == "ok", "ccn"])
zcta = read(RAW / "PLACES" / "places_zcta.csv", usecols=["ZCTA5", "Geolocation"])
zcta_pt = zcta.set_index(zcta.ZCTA5.str.zfill(5)).Geolocation.str.extract(r"POINT \((\S+) (\S+)\)").astype(float)
need = facilities[(facilities.facility_class == "hospital") & ~facilities.ccn.isin(placed) & facilities.zip.isin(zcta_pt.index)]
need = need.assign(lon=zcta_pt.loc[need.zip, 0].values, lat=zcta_pt.loc[need.zip, 1].values)

osm_path, osm = json_cache("openstreetmap.json")
osm.setdefault("places", {}); osm.setdefault("searched", [])
searched = set(osm["searched"])
todo = need[~need.ccn.isin(searched)]
for start in range(0, len(todo), 60):
    chunk = todo.iloc[start:start + 60]
    for place in osm_hospitals_near(list(zip(chunk.lat, chunk.lon, [12] * len(chunk))), log=print):
        osm["places"][place["osm"]] = place
    searched.update(chunk.ccn)
    osm["searched"] = sorted(searched)
    osm_path.write_text(json.dumps(osm))  # save as we go so a rerun resumes
    print(f"  OpenStreetMap: {start + len(chunk)}/{len(todo)} searched")
    time.sleep(2)

places = list(osm["places"].values())
matches = {r.ccn: best_osm_match({"name": r.name, "lat": r.lat, "lon": r.lon}, places, 12) for r in need.itertuples()}
claims = {}
for ccn, m in matches.items():
    if m:
        claims.setdefault(m[3]["osm"], []).append(ccn)
names = facilities.set_index("ccn").name
for ccns in claims.values():  # one OSM hospital can't be two differently named hospitals
    if len({frozenset(name_tokens(names[c])) for c in ccns}) > 1:
        keep = max(ccns, key=lambda c: (matches[c][0], -matches[c][1]))
        for c in ccns:
            if c != keep:
                matches[c] = None

# Accuracy check: the OSM point must fall in CMS's county (or state when POS has no county).
cnty_path, cnty_cache = json_cache("point_counties.json")
def point_county(place):
    d = get_json(COORDS, {**GEO_PARAMS, "x": place["lon"], "y": place["lat"]})
    counties = (d or {}).get("result", {}).get("geographies", {}).get("Counties", [])
    return counties[0]["GEOID"] if counties else ""
todo = {m[3]["osm"]: m[3] for m in matches.values() if m and m[3]["osm"] not in cnty_cache}
with ThreadPoolExecutor(max_workers=8) as pool:
    cnty_cache.update(zip(todo, pool.map(point_county, todo.values())))
cnty_path.write_text(json.dumps(cnty_cache))

fac_county = facilities.set_index("ccn")
osm_rows = []
for ccn, m in matches.items():
    if not m:
        continue
    score, d, osm_name, place = m
    f = fac_county.loc[ccn]
    county = cnty_cache.get(place["osm"], "")
    ok_place = county == f.county_fips if f.county_fips and f.state_fips != "09" else county[:2] == f.state_fips
    # A word-for-word name match may sit just over a county line (Mercy Hospital Joplin: Jasper/Newton).
    ok_place = ok_place or (score == 1.0 and county[:2] == f.state_fips)
    osm_rows.append({"ccn": ccn, "source": "osm", "street": osm_name, "lat": place["lat"], "lon": place["lon"],
                     "match_status": "Match", "match_type": "OSM_name",
                     "matched_address": f"{osm_name} (OpenStreetMap {place['osm']})",
                     "verdict": "ok" if ok_place else "wrong_county", "addr_id": "osm:" + place["osm"]})
osm_rows = pd.DataFrame(osm_rows)
if len(osm_rows):
    g = pd.concat([g, osm_rows], ignore_index=True)
print(f"Hospitals still unplaced: {len(need):,}   OSM name matches: {len(osm_rows):,}   "
      f"in CMS's county: {(osm_rows.verdict == 'ok').sum() if len(osm_rows) else 0:,}")

### Pick one accurate location per facility; everything else is an outlier

Among the accurate candidates, `Exact` matches win, then HGI over POS over enrollment over OpenStreetMap. `n_accurate_sources` counts how many independent addresses agreed. `source_spread_mi` is the largest distance between accurate candidates; a big spread usually means the CMS files disagree about the campus.

In [ ]:
def miles(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = np.sin((p2 - p1) / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    return 2 * 3958.8 * np.arcsin(np.sqrt(np.minimum(1, a)))

ok = g[g.verdict == "ok"].copy()
ok["rank"] = (ok.match_type != "Exact").astype(int) * 10 + ok.source.map({"hgi": 0, "pos": 1, "enrollment": 2, "osm": 3})
ok = ok.sort_values(["ccn", "rank"])
best = ok.drop_duplicates("ccn").set_index("ccn")
first = best[["lat", "lon"]].rename(columns={"lat": "lat0", "lon": "lon0"})
spread = ok.join(first, on="ccn").assign(d=lambda d: miles(d.lat, d.lon, d.lat0, d.lon0)).groupby("ccn").d.max()

located = facilities.set_index("ccn").join(best[["lat", "lon", "source", "match_type", "matched_address"]], how="inner")
located["geocode_source"] = located.pop("source")
located["n_accurate_sources"] = ok.drop_duplicates(["ccn", "source"]).groupby("ccn").size()
located["source_spread_mi"] = spread.round(2)
located = located.reset_index()

# Outliers: why each facility has no accurate coordinate (its best failure reason)
FAIL_ORDER = ["wrong_county", "wrong_state", "non_exact_house_number", "non_exact_direction", "tie", "no_match"]
failed = g[~g.ccn.isin(located.ccn)].copy()
failed["order"] = failed.verdict.map({v: i for i, v in enumerate(FAIL_ORDER)}).fillna(len(FAIL_ORDER))
reason = failed.sort_values("order").groupby("ccn").agg(
    reason=("verdict", "first"), addresses_tried=("addr_id", "nunique"),
    best_attempt=("matched_address", lambda s: next((x for x in s if x), "")))
outliers = facilities[~facilities.ccn.isin(located.ccn)].set_index("ccn").join(reason).reset_index()
outliers["reason"] = outliers.reason.fillna("no_usable_address")
outliers["addresses_tried"] = outliers.addresses_tried.fillna(0).astype(int)

for cls in ["hospital", "other"]:
    n = (facilities.facility_class == cls).sum()
    k = (located.facility_class == cls).sum()
    print(f"{cls:9s}: {k:,} / {n:,} accurately located ({k / n:.2%}), {n - k:,} outliers")
hgi_ok = located[located.in_hgi].shape[0]
print(f"HGI hospitals: {hgi_ok:,} / {len(hgi):,} located ({1 - hgi_ok / len(hgi):.2%} outliers, vs 13.69% first pass)")
print("\nOutlier reasons:\n" + outliers.groupby(["facility_class", "reason"]).size().to_string())
print("\nLocated via:\n" + located.groupby(["geocode_source", "match_type"]).size().to_string())

### Write the location CSVs

These are the files to use for hospital locations. Coordinates are WGS84 (`lat`, `lon`).
* `hospital_locations.csv`: hospitals with an accurate coordinate. `counts_for_coverage` is True for acute care, critical access, VA, DoD and rural emergency hospitals listed in HGI; the web scorer uses it.
* `other_facility_locations.csv`: FQHCs and Rural Health Clinics with an accurate coordinate.
* `facility_location_outliers.csv`: everything we could **not** place accurately, with the reason. These are excluded from the map.

In [ ]:
COVERAGE_TYPES = {"Acute Care Hospitals", "Critical Access Hospitals", "Acute Care - Veterans Administration",
                  "Acute Care - Department of Defense", "Rural Emergency Hospital"}
located["counts_for_coverage"] = (located.facility_class == "hospital") & located.type.isin(COVERAGE_TYPES) & located.in_hgi
located["lat"] = located.lat.round(6)
located["lon"] = located.lon.round(6)

cols = ["ccn", "name", "type", "facility_class", "address", "city", "state", "zip", "county", "state_fips", "county_fips",
        "phone", "ownership", "emergency", "rating", "beds", "counts_for_coverage", "in_hgi", "lat", "lon",
        "geocode_source", "match_type", "matched_address", "n_accurate_sources", "source_spread_mi"]
located = located.sort_values(["facility_class", "state", "ccn"])
located[located.facility_class == "hospital"][cols].to_csv(OUT / "hospital_locations.csv", index=False)
located[located.facility_class == "other"][cols].to_csv(OUT / "other_facility_locations.csv", index=False)
outliers[["ccn", "name", "type", "facility_class", "address", "city", "state", "zip", "in_hgi",
          "reason", "addresses_tried", "best_attempt"]].sort_values(["facility_class", "state", "ccn"]).to_csv(
    OUT / "facility_location_outliers.csv", index=False)
for f in sorted(OUT.glob("*.csv")):
    print(f"{f.relative_to(ROOT)}: {sum(1 for _ in open(f, encoding='utf-8')) - 1:,} rows")